In [3]:
import numpy as np

class MSELoss:
    def compute_loss(self,y,F):
        difference = y-F
        squared_difference = difference**2
        mean_squared = np.mean(squared_difference)
        return 0.5 * mean_squared

    def compute_gradient(self,y,F):
        return y-F

In [4]:
class RegressionStump:
    def __init__(self):
        self.split_feature = None
        self.split_threshold = None
        self.left_value = None
        self.right_value = None

    def fit(self,X,y):
        best_loss = float('inf')
        for j in range(X.shape[1]):
            
            unique_feature_values = np.unique(X[:,j])
            for t in unique_feature_values:
                X_left = X[X[:,j]<=t]
                y_left = y[X[:,j]<=t]
                X_right = X[X[:,j]>t]
                y_right = y[X[:,j]>t]

                if len(y_left)==0 or len(y_right) ==0:
                    continue

                mse = MSELoss()
                left_loss = mse.compute_loss(y_left,np.mean(y_left))
                right_loss = mse.compute_loss(y_right,np.mean(y_right))

                left_weight = len(y_left)/len(y)
                right_weight = len(y_right)/len(y)

                total_loss = left_weight * left_loss + right_weight * right_loss

                if(best_loss>total_loss):
                    best_loss = total_loss
                    self.split_feature = j
                    self.split_threshold = t
                    self.left_value = np.mean(y_left)
                    self.right_value = np.mean(y_right)

    def predict(self,X):
        predictions = np.zeros((X.shape[0],1))
        predictions[X[:,self.split_feature]<=self.split_threshold] = self.left_value
        predictions[X[:,self.split_feature]>self.split_threshold] = self.right_value
        return predictions    

In [5]:
class GradientBoostingRegressor:
    def __init__(self, n_estimators=100, learning_rate=0.1):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.models = []
        self.F0 = None

    def fit(self,X,y):
        self.F0 = np.mean(y)
        mse = MSELoss()
        F = np.full((X.shape[0],1),self.F0)
        for i in range(self.n_estimators):
            r_m = mse.compute_gradient(y,F)
            regression_stump = RegressionStump()
            regression_stump.fit(X,r_m)
            self.models.append(regression_stump)
            F = F + self.learning_rate * regression_stump.predict(X)

    def predict(self,X):
        predictions = np.full((X.shape[0],1),self.F0)
        for model in self.models:
            predictions = predictions + self.learning_rate * model.predict(X)

        return predictions

In [9]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor as SklearnGBR

# 1. Generate Synthetic Data (A noisy sine wave)
np.random.seed(42)
n_samples = 200
X = np.linspace(0, 10, n_samples).reshape(-1, 1) # Shape: (200, 1)
y = np.sin(X) + np.random.normal(0, 0.2, (n_samples, 1)) # Shape: (200, 1)

# 2. Train YOUR From-Scratch Model
# (Assuming you applied the shape fixes to RegressionStump)
my_gbm = GradientBoostingRegressor(n_estimators=10000, learning_rate=0.1)
my_gbm.fit(X, y)
my_predictions = my_gbm.predict(X)

# 3. Train Scikit-Learn's Model (for comparison)
# We set max_depth=1 to match our Decision Stump!
sk_gbm = SklearnGBR(n_estimators=10000, learning_rate=0.1, max_depth=1, random_state=42)
sk_gbm.fit(X, y.ravel()) # Sklearn expects y to be (n,)
sk_predictions = sk_gbm.predict(X).reshape(-1, 1)

# 4. Compare Results
print(f"My GBM MSE:      {np.mean((y - my_predictions)**2):.4f}")
print(f"Sklearn GBM MSE: {np.mean((y - sk_predictions)**2):.4f}")

# Optional: Plot the results to visually confirm!
plt.scatter(X, y, color='lightgray', label='True Data')
plt.plot(X, my_predictions, color='red', label='My GBM', linewidth=2)
plt.plot(X, sk_predictions, color='blue', linestyle='dashed', label='Sklearn GBM', linewidth=2)
plt.legend()
plt.title("Gradient Boosting: Scratch vs Scikit-Learn")
plt.show()

KeyboardInterrupt: 